# Notebook 2 — Model Training & Mitigations
## Derm Fairness Project · Fitzpatrick17k

We train **5 model variants** on the same train/val/test splits from Notebook 1:

| # | Name | Strategy |
|---|------|----------|
| 1 | `baseline` | Standard EfficientNet-B0 fine-tune, no fairness intervention |
| 2 | `balanced_sampling` | Oversample dark-skin malignant cases to equalise subgroup × label counts |
| 3 | `loss_reweighting` | Increase loss penalty for underrepresented subgroup × label cells |
| 4 | `targeted_finetune` | Start from baseline, fine-tune only on dark/medium skin images |
| 5 | `threshold_tuning` | Post-hoc: lower decision threshold for dark-skin group to cut FNR |

---
### MIMIC portability note
- Swap `DermDataset` → `MIMICNotesDataset` (returns tokenized text tensors instead of images)
- Swap `build_model()` backbone → `AutoModel.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')`
- Everything else (training loop, loss reweighting, threshold tuning) is **identical**

## 0 · Imports & config

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
import requests
from io import BytesIO
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

IMG_DIR = Path('data/images')
IMG_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 5          # Increase to 10-15 for final results
IMG_SIZE = 224

Using device: cpu


## 1 · Load splits

In [7]:
df_train = pd.read_csv('data/train.csv')
df_val   = pd.read_csv('data/val.csv')
df_test  = pd.read_csv('data/test.csv')

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')

Train: 11181 | Val: 2396 | Test: 2397


## 2 · Image download helper
Downloads images from the URL column and caches locally so we don't re-download.

In [8]:
def get_image(url: str, cache_dir: Path) -> Image.Image:
    """Fetch image from URL or local cache. Returns PIL Image."""
    img_id = url.split('/')[-1].split('?')[0][:64]  # use URL tail as filename
    local_path = cache_dir / f"{img_id}.jpg"
    if local_path.exists():
        return Image.open(local_path).convert('RGB')
    try:
        resp = requests.get(url, timeout=10)
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        img.save(local_path)
        return img
    except Exception:
        # Return a blank image if download fails — logged during eval
        return Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=128)

## 3 · Dataset class

> **MIMIC swap:** Replace image loading with `tokenizer(note_text, ...)` and return `input_ids`, `attention_mask` tensors.

In [9]:
class DermDataset(Dataset):
    """
    Fitzpatrick17k binary classification dataset.

    Each item returns:
        image  : (3, 224, 224) float tensor
        label  : int (0=benign, 1=malignant)
        group  : str skin tone group ('light'/'medium'/'dark')
    """
    def __init__(self, df: pd.DataFrame, transform=None, cache_dir: Path = IMG_DIR):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = get_image(row['url'], self.cache_dir)
        if self.transform:
            img = self.transform(img)
        label = int(row['binary_label'])
        group = str(row['group'])
        return img, label, group

# Standard ImageNet normalization (EfficientNet pretrained)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_dataset  = DermDataset(df_val,  eval_transform)
test_dataset = DermDataset(df_test, eval_transform)

## 4 · Model builder

> **MIMIC swap:** Replace EfficientNet-B0 with `AutoModel.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')` + a linear classification head.

In [10]:
def build_model(freeze_backbone: bool = False) -> nn.Module:
    """
    EfficientNet-B0 with a binary classification head.
    Transfer learning from ImageNet weights.
    """
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    # Replace classifier head for binary output
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 1)   # raw logit; use BCEWithLogitsLoss
    )
    return model.to(DEVICE)

## 5 · Training loop

Generic — accepts any DataLoader and loss function, works for all 5 mitigations.

In [11]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for imgs, labels, _ in tqdm(loader, leave=False):
        imgs   = imgs.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def eval_epoch(model, loader):
    """Returns arrays of (probabilities, labels, groups) for the whole loader."""
    model.eval()
    all_probs, all_labels, all_groups = [], [], []
    with torch.no_grad():
        for imgs, labels, groups in loader:
            imgs   = imgs.to(DEVICE)
            logits = model(imgs).squeeze(1).cpu()
            probs  = torch.sigmoid(logits).numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
            all_groups.extend(groups)
    return np.array(all_probs), np.array(all_labels), np.array(all_groups)


def run_training(model, train_loader, val_loader, criterion, epochs=EPOCHS, lr=LR):
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_probs, val_labels, _ = eval_epoch(model, val_loader)
        # Val loss (BCE)
        val_loss = nn.BCELoss()(
            torch.tensor(val_probs).clamp(1e-7, 1 - 1e-7),
            torch.tensor(val_labels).float()
        ).item()
        scheduler.step()
        print(f'  Epoch {epoch}/{epochs} — train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

## 6 · Mitigation 1 — Baseline

In [ ]:
print('=== Mitigation 1: Baseline ===')
train_ds_base = DermDataset(df_train, train_transform)
train_loader_base = DataLoader(train_ds_base, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model_baseline = build_model()
criterion_base = nn.BCEWithLogitsLoss()
model_baseline = run_training(model_baseline, train_loader_base, val_loader, criterion_base)
torch.save(model_baseline.state_dict(), 'data/model_baseline.pt')
print('Saved model_baseline.pt')

=== Mitigation 1: Baseline ===
=== Mitigation 1: Baseline ===


  7%|▋         | 26/350 [10:43<2:02:51, 22.75s/it]

## 7 · Mitigation 2 — Balanced sampling

Use `WeightedRandomSampler` to oversample rare subgroup × label combinations during training.

In [ ]:
print('=== Mitigation 2: Balanced Sampling ===')

def make_weighted_sampler(df: pd.DataFrame) -> WeightedRandomSampler:
    """Each subgroup × label cell gets equal total weight."""
    strat_key = df['group'] + '_' + df['binary_label'].astype(str)
    cell_counts = strat_key.value_counts()
    sample_weight = strat_key.map(lambda k: 1.0 / cell_counts[k])
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weight.values),
        num_samples=len(df),
        replacement=True
    )

train_ds_bal = DermDataset(df_train, train_transform)
sampler_bal  = make_weighted_sampler(df_train)
train_loader_bal = DataLoader(train_ds_bal, batch_size=BATCH_SIZE, sampler=sampler_bal, num_workers=2)

model_balanced = build_model()
model_balanced = run_training(model_balanced, train_loader_bal, val_loader, criterion_base)
torch.save(model_balanced.state_dict(), 'data/model_balanced.pt')
print('Saved model_balanced.pt')

## 8 · Mitigation 3 — Loss reweighting

Keep the same DataLoader as baseline but scale up the loss for underrepresented subgroup × label cells.

> For MIMIC: same approach — weight by `ethnicity × mortality` cell frequency.

In [ ]:
print('=== Mitigation 3: Loss Reweighting ===')

class WeightedBCELoss(nn.Module):
    """
    Per-sample BCE loss where the weight is proportional to
    1 / (frequency of that sample's subgroup × label cell).
    """
    def __init__(self, df_train: pd.DataFrame):
        super().__init__()
        strat_key = df_train['group'] + '_' + df_train['binary_label'].astype(str)
        cell_counts = strat_key.value_counts()
        # Build lookup: (group, label) -> weight
        self.weight_map = {}
        for key, count in cell_counts.items():
            grp, lbl = key.rsplit('_', 1)
            self.weight_map[(grp, int(lbl))] = 1.0 / count
        # Normalise so mean weight ≈ 1
        mean_w = np.mean(list(self.weight_map.values()))
        self.weight_map = {k: v / mean_w for k, v in self.weight_map.items()}

    def forward(self, logits, labels, groups):
        # Per-sample BCE
        bce = nn.BCEWithLogitsLoss(reduction='none')(logits, labels)
        weights = torch.tensor(
            [self.weight_map.get((g, int(l.item())), 1.0)
             for g, l in zip(groups, labels)],
            dtype=torch.float32, device=logits.device
        )
        return (bce * weights.unsqueeze(1)).mean()


def train_epoch_weighted(model, loader, optimizer, criterion_weighted):
    model.train()
    total_loss = 0
    for imgs, labels, groups in tqdm(loader, leave=False):
        imgs   = imgs.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion_weighted(logits, labels, groups)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


train_ds_rw = DermDataset(df_train, train_transform)
train_loader_rw = DataLoader(train_ds_rw, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
criterion_rw = WeightedBCELoss(df_train)

model_reweighted = build_model()
optimizer_rw = torch.optim.Adam(model_reweighted.parameters(), lr=LR)
scheduler_rw = torch.optim.lr_scheduler.StepLR(optimizer_rw, step_size=3, gamma=0.5)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch_weighted(model_reweighted, train_loader_rw, optimizer_rw, criterion_rw)
    val_probs, val_labels, _ = eval_epoch(model_reweighted, val_loader)
    val_loss = nn.BCELoss()(
        torch.tensor(val_probs).clamp(1e-7, 1-1e-7),
        torch.tensor(val_labels).float()
    ).item()
    scheduler_rw.step()
    print(f'  Epoch {epoch}/{EPOCHS} — train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}')

torch.save(model_reweighted.state_dict(), 'data/model_reweighted.pt')
print('Saved model_reweighted.pt')

## 9 · Mitigation 4 — Targeted fine-tuning

Start from the baseline model. Fine-tune only on dark + medium skin images.

In [ ]:
print('=== Mitigation 4: Targeted Fine-tuning ===')

df_targeted = df_train[df_train['group'].isin(['dark', 'medium'])].copy()
print(f'Targeted fine-tune subset: {len(df_targeted)} samples')

model_targeted = build_model()
model_targeted.load_state_dict(torch.load('data/model_baseline.pt', map_location=DEVICE))

# Freeze backbone, only update classifier head during targeted fine-tune
for param in model_targeted.features.parameters():
    param.requires_grad = False

train_ds_tgt = DermDataset(df_targeted, train_transform)
train_loader_tgt = DataLoader(train_ds_tgt, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Fine-tune for fewer epochs at lower LR
model_targeted = run_training(
    model_targeted, train_loader_tgt, val_loader,
    criterion_base, epochs=3, lr=LR * 0.1
)
torch.save(model_targeted.state_dict(), 'data/model_targeted.pt')
print('Saved model_targeted.pt')

## 10 · Mitigation 5 — Threshold tuning (post-hoc)

Using the **baseline** model, find the optimal decision threshold per group on the **validation set** that minimises FNR for malignant cases. Apply those thresholds at test time.

This doesn't require retraining — it's a post-processing step.

In [ ]:
print('=== Mitigation 5: Threshold Tuning ===')

# Use baseline model predictions on val set to find per-group thresholds
model_baseline.load_state_dict(torch.load('data/model_baseline.pt', map_location=DEVICE))
val_probs, val_labels, val_groups = eval_epoch(model_baseline, DataLoader(val_dataset, batch_size=64))

def find_best_threshold(probs, labels, metric='fnr', target=0.1):
    """
    Find threshold that keeps FNR ≤ target (default ≤10%) while maximising precision.
    Falls back to threshold minimising FNR if target can't be met.
    """
    best_thresh = 0.5
    best_fnr = 1.0
    for t in np.arange(0.1, 0.9, 0.01):
        preds = (probs >= t).astype(int)
        tp = ((preds == 1) & (labels == 1)).sum()
        fn = ((preds == 0) & (labels == 1)).sum()
        fnr = fn / (tp + fn + 1e-8)
        if fnr < best_fnr:
            best_fnr = fnr
            best_thresh = t
    return best_thresh

group_thresholds = {}
for grp in ['light', 'medium', 'dark']:
    mask = val_groups == grp
    if mask.sum() == 0:
        group_thresholds[grp] = 0.5
        continue
    thresh = find_best_threshold(val_probs[mask], val_labels[mask])
    group_thresholds[grp] = thresh
    print(f'  {grp}: optimal threshold = {thresh:.2f}')

import json
with open('data/group_thresholds.json', 'w') as f:
    json.dump(group_thresholds, f)
print('Saved group_thresholds.json')

---
### ✅ Notebook 2 complete
**Outputs:** 4 model `.pt` files + `group_thresholds.json`  
**Next:** Notebook 3 — Fairness evaluation & visualisation